# 00 — Setup and sanity checks

This notebook is the prerequisite for everything else in the harness. It does four things:

1. Verifies that the required environment variables are set (`ANTHROPIC_API_KEY`, `NEO4J_PASSWORD`, `ARANGO_PASSWORD`).
2. Opens a connection to graphonauts2's PostgreSQL, the local Neo4j, and the local ArangoDB. Each connection runs a tiny smoke query so we fail fast if a service is down or empty.
3. Confirms the LDBC SNB data is actually loaded — empty databases produce silently-zero execution metrics later, which is worse than an exception now.
4. Prints a one-page summary of the gold-standard dataset so you can see what `01_translation_run` will iterate over.

Run this first. If anything fails, fix the environment before launching the LLM-heavy `01_translation_run.ipynb` — that notebook makes ~56 paid Anthropic calls and you do not want to discover a bad config midway through.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import yaml

# Resolve repo root from the notebook location: evaluation/notebooks/00_setup.ipynb
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate repo root"

DATASETS_DIR = REPO_ROOT / "evaluation" / "datasets"
OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Datasets:  {DATASETS_DIR}")
print(f"Outputs:   {OUTPUTS_DIR}")

## Environment variables

Each variable is checked individually so the failure message names exactly what is missing. We do **not** print the value of any secret — only whether it was set.

In [ ]:
REQUIRED_ENV = ["ANTHROPIC_API_KEY", "NEO4J_PASSWORD", "ARANGO_PASSWORD"]
missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
for name in REQUIRED_ENV:
    status = "set" if os.environ.get(name) else "MISSING"
    print(f"  {name}: {status}")
if missing:
    raise RuntimeError(f"Set these before continuing: {missing}")

## PostgreSQL — graphonauts2's LDBC instance

Connection settings match `graphonauts2/docker/postgres.compose.yml`: port 5433 (the compose remaps to avoid colliding with Homebrew Postgres), user `graphonaut`, db `graphonaut`, default password `password`.

We run `SELECT count(*) FROM person` because Person is the most heavily-referenced LDBC table — if it has zero rows, downstream execution metrics will be useless.

In [ ]:
import psycopg

PG_DSN = (
    f"host=localhost port=5433 "
    f"user=graphonaut password={os.environ.get('POSTGRES_PASSWORD', 'password')} "
    f"dbname=graphonaut"
)

with psycopg.connect(PG_DSN) as conn, conn.cursor() as cur:
    cur.execute("SELECT version()")
    version = cur.fetchone()[0].split(",")[0]
    print(f"Postgres: {version}")
    counts: dict[str, int] = {}
    for table in ("person", "forum", "post", "comment", "tag", "knows", "forum_has_member", "likes_post"):
        cur.execute(f"SELECT count(*) FROM {table}")
        counts[table] = cur.fetchone()[0]
for t, n in counts.items():
    print(f"  {t:>20}: {n:>10,} rows")
if counts["person"] == 0:
    raise RuntimeError("Postgres `person` table is empty — load LDBC data before running execution metrics.")

## Neo4j — the Cypher execution target

Connection settings match `config/servers/neo4j.yaml` (bolt://localhost:7687, user `neo4j`). We count `(:Person)` to mirror the Postgres check above.

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, os.environ["NEO4J_PASSWORD"]))
try:
    with driver.session(database=NEO4J_DB) as session:
        person_n = session.run("MATCH (p:Person) RETURN count(p) AS n").single()["n"]
        labels = session.run("CALL db.labels() YIELD label RETURN collect(label) AS labels").single()["labels"]
        print(f"Neo4j: {len(labels)} labels — {sorted(labels)}")
        print(f"  Person nodes: {person_n:,}")
finally:
    driver.close()
if person_n == 0:
    raise RuntimeError("Neo4j has no Person nodes — load LDBC data before running execution metrics.")

## ArangoDB — the AQL execution target

Connection settings match `config/servers/arangodb.yaml`. `ldbc` is the database name graphonauts2 loads into. The `graph_name` is also `ldbc`.

In [ ]:
from arango.client import ArangoClient

ARANGO_URL = os.environ.get("ARANGO_URL", "http://localhost:8529")
ARANGO_USER = os.environ.get("ARANGO_USER", "root")
ARANGO_DB = os.environ.get("ARANGO_DATABASE", "ldbc")

client = ArangoClient(hosts=ARANGO_URL)
db = client.db(ARANGO_DB, username=ARANGO_USER, password=os.environ["ARANGO_PASSWORD"])
collections = sorted(c["name"] for c in db.collections() if not c["name"].startswith("_"))
print(f"ArangoDB: {len(collections)} collections — {collections}")
person_count = db.collection("Person").count() if "Person" in collections else 0
print(f"  Person docs: {person_count:,}")
if person_count == 0:
    raise RuntimeError("ArangoDB has no Person documents — load LDBC data before running execution metrics.")

## Gold-standard dataset summary

What `01_translation_run` will iterate over: every query in `tpch.yaml` and `ldbc.yaml`, for both dialects (Cypher and AQL). Total work = `2 × (|tpch| + |ldbc|)` LLM-driven translations.

In [ ]:
import pandas as pd

rows = []
for path in sorted(DATASETS_DIR.glob("*.yaml")):
    data = yaml.safe_load(path.read_text())
    for q in data["queries"]:
        rows.append({
            "schema": data["schema"],
            "id": q["id"],
            "difficulty": q["difficulty"],
            "sql_features": ", ".join(q.get("sql_features", [])) or "—",
            "description": q["description"],
        })
dataset_df = pd.DataFrame(rows)
print(f"Total gold queries: {len(dataset_df)}")
print(f"Total translations to run (× 2 dialects): {len(dataset_df) * 2}")
dataset_df.pivot_table(index="schema", columns="difficulty", values="id", aggfunc="count", fill_value=0)

In [ ]:
dataset_df.head(20)

All checks passed. Continue to `01_translation_run.ipynb`.